# Task 1: Formal Game Representation

In [ ]:
import copy
MAX = 'X'
MIN = 'O'
EMPTY = ' '
def getInitialState():
    """Returns an empty 3x3 board represented as a list of lists."""
    return [[EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY]]

# 2. Current Player
def getPlayer(state):
    """Returns MAX ('X') or MIN ('O') based on whose turn it is."""
    x_count = sum(row.count(MAX) for row in state)
    o_count = sum(row.count(MIN) for row in state)
    # MAX goes first; equal counts means it is MAX's turn
    return MAX if x_count == o_count else MIN

# 3. Available Actions
def getActions(state):
    """Returns list of empty cell positions (i, j) as possible moves."""
    actions = []
    for i in range(3):
        for j in range(3):
            if state[i][j] == EMPTY:
                actions.append((i, j))
    return actions

# 4. Result Function
def getResult(state, action):
    """Returns a new state after the current player applies the given action."""
    new_state = copy.deepcopy(state)
    i, j = action
    new_state[i][j] = getPlayer(state)
    return new_state

def checkWinner(state):
    """Returns MAX, MIN, or None depending on who (if anyone) has won."""
    for row in state:
        if row[0] == row[1] == row[2] and row[0] != EMPTY:
            return row[0]
    for j in range(3):
        if state[0][j] == state[1][j] == state[2][j] and state[0][j] != EMPTY:
            return state[0][j]
    if state[0][0] == state[1][1] == state[2][2] and state[0][0] != EMPTY:
        return state[0][0]
    if state[0][2] == state[1][1] == state[2][0] and state[0][2] != EMPTY:
        return state[0][2]
    return None

# 5. Terminal Test
def isTerminal(state):
    """Returns True if game is over (win or draw)."""
    if checkWinner(state) is not None:
        return True
    return all(state[i][j] != EMPTY for i in range(3) for j in range(3))

# 6. Utility Function
def getUtility(state):
    """Returns +1 if MAX wins, -1 if MIN wins, 0 for draw."""
    winner = checkWinner(state)
    if winner == MAX:
        return 1
    elif winner == MIN:
        return -1
    return 0

# Helper: pretty print board
def printBoard(state):
    print("\n  0   1   2")
    for i, row in enumerate(state):
        print(f"{i} " + " | ".join(row))
        if i < 2:
            print("  ---------")
    print()

print("=== Task 1: Formal Game Representation ===")

state = getInitialState()
print("Initial Board:")
printBoard(state)
print(f"Current Player : {getPlayer(state)}")
print(f"Available Moves: {getActions(state)}")
print(f"Is Terminal    : {isTerminal(state)}")

# Apply a few moves to show state transitions
state = getResult(state, (1, 1))   # X centre
state = getResult(state, (0, 0))   # O top-left
state = getResult(state, (0, 2))   # X top-right
print("After 3 moves (X played (1,1) and (0,2); O played (0,0)):")
printBoard(state)
print(f"Current Player : {getPlayer(state)}")
print(f"Is Terminal    : {isTerminal(state)}")
print(f"Utility        : {getUtility(state)}")

# Demonstrate terminal detection on a win
win_state = [['X', 'X', 'X'],
             ['O', 'O', ' '],
             [' ', ' ', ' ']]
print("Win state (X wins top row):")
printBoard(win_state)
print(f"Is Terminal: {isTerminal(win_state)}")
print(f"Utility    : {getUtility(win_state)}")

=== Task 1: Formal Game Representation ===
Initial Board:

  0   1   2
0   |   |  
  ---------
1   |   |  
  ---------
2   |   |  

Current Player : X
Available Moves: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
Is Terminal    : False
After 3 moves (X played (1,1) and (0,2); O played (0,0)):

  0   1   2
0 O |   | X
  ---------
1   | X |  
  ---------
2   |   |  

Current Player : O
Is Terminal    : False
Utility        : 0
Win state (X wins top row):

  0   1   2
0 X | X | X
  ---------
1 O | O |  
  ---------
2   |   |  

Is Terminal: True
Utility    : 1


# Task 2: Game Tree Generation

In [2]:
node_count = 0
max_depth = 0

def generateTree(state, depth=0):
    """Recursively generates the complete game tree and counts nodes/depth."""
    global node_count, max_depth

    # Increment node count for each node visited
    node_count += 1
    if depth > max_depth:
        max_depth = depth
    if isTerminal(state):
        return {"state": state, "depth": depth, "children": [], "utility": getUtility(state)}
    children = []
    for action in getActions(state):
        child_state = getResult(state, action)
        child_node = generateTree(child_state, depth + 1)
        children.append({"action": action, "node": child_node})

    return {"state": state, "depth": depth, "children": children}


# Driver Code
print("=== Task 2: Game Tree Generation ===")
print("Generating complete game tree from initial state...\n")

node_count = 0
max_depth = 0

initial = getInitialState()
tree = generateTree(initial)

print(f"Total Nodes Generated : {node_count}")
print(f"Maximum Depth of Tree : {max_depth}")
print()
print("Note: Full tree has fewer nodes than 9! = 362,880 because many")
print("branches terminate early at win/draw states.")

=== Task 2: Game Tree Generation ===
Generating complete game tree from initial state...

Total Nodes Generated : 549946
Maximum Depth of Tree : 9

Note: Full tree has fewer nodes than 9! = 362,880 because many
branches terminate early at win/draw states.


# Task 3: Minimax Algorithm

In [4]:
nodes_explored = 0

def maxValue(state):
    """Returns the maximum utility achievable from this state (MAX's turn)."""
    global nodes_explored
    nodes_explored += 1

    if isTerminal(state):
        return getUtility(state)

    v = float('-inf')
    for action in getActions(state):
        v = max(v, minValue(getResult(state, action)))
    return v


def minValue(state):
    """Returns the minimum utility achievable from this state (MIN's turn)."""
    global nodes_explored
    nodes_explored += 1

    if isTerminal(state):
        return getUtility(state)

    v = float('inf')
    for action in getActions(state):
        v = min(v, maxValue(getResult(state, action)))
    return v


def minimaxDecision(state):
    """Selects the action that maximises utility for MAX player."""
    best_action = None
    best_value = float('-inf')

    for action in getActions(state):
        value = minValue(getResult(state, action))
        if value > best_value:
            best_value = value
            best_action = action

    return best_action, best_value


# Driver
print("=== Task 3: Minimax Algorithm ===")
nodes_explored = 0
initial = getInitialState()

print("Running Minimax from initial (empty) board...")
action, value = minimaxDecision(initial)

print(f"Best Move      : {action}")
print(f"Minimax Value  : {value}")
print(f"Nodes Explored : {nodes_explored}")

# Also test on a mid-game state to show correctness
print()
mid_state = [['X', 'O', 'X'],
             ['O', 'X', ' '],
             [' ', ' ', 'O']]
print("Mid-game board (X to move, should win):")
printBoard(mid_state)
nodes_explored = 0
action2, value2 = minimaxDecision(mid_state)
print(f"Best Move      : {action2}")
print(f"Minimax Value  : {value2}")
print(f"Nodes Explored : {nodes_explored}")

=== Task 3: Minimax Algorithm ===
Running Minimax from initial (empty) board...
Best Move      : (0, 0)
Minimax Value  : 0
Nodes Explored : 549945

Mid-game board (X to move, should win):

  0   1   2
0 X | O | X
  ---------
1 O | X |  
  ---------
2   |   | O

Best Move      : (2, 0)
Minimax Value  : 1
Nodes Explored : 11


# Task 4: Depth-Limited Minimax

In [5]:
DEPTH_LIMIT = 2  # Default; overridden in experimental loop below

# ── Heuristic Evaluation Function ─────────────────────────────────────────
def evaluateHeuristic(state):
    """
    Evaluates all rows, columns, and diagonals using the scoring scheme:
      +100 : three MAX (X X X)
      +10  : two MAX + one empty
      +1   : one MAX + two empty
      -100 : three MIN (O O O)
      -10  : two MIN + one empty
      -1   : one MIN + two empty
    Returns the sum over all 8 lines.
    """
    def scoreLine(cells):
        x_cnt = cells.count(MAX)
        o_cnt = cells.count(MIN)
        e_cnt = cells.count(EMPTY)
        if x_cnt > 0 and o_cnt > 0:  # blocked line
            return 0
        if x_cnt == 3:                return  100
        if x_cnt == 2 and e_cnt == 1: return   10
        if x_cnt == 1 and e_cnt == 2: return    1
        if o_cnt == 3:                return -100
        if o_cnt == 2 and e_cnt == 1: return  -10
        if o_cnt == 1 and e_cnt == 2: return   -1
        return 0
    total = 0
    for row in state:                                       # rows
        total += scoreLine(row)
    for j in range(3):                                     # columns
        total += scoreLine([state[i][j] for i in range(3)])
    total += scoreLine([state[i][i] for i in range(3)])    # main diagonal
    total += scoreLine([state[i][2 - i] for i in range(3)]) # anti-diagonal
    return total
def maxValueDepth(state, depth):
    if isTerminal(state):
        return getUtility(state) * 1000   # scale so terminal > heuristic
    if depth == DEPTH_LIMIT:
        return evaluateHeuristic(state)

    v = float('-inf')
    for action in getActions(state):
        v = max(v, minValueDepth(getResult(state, action), depth + 1))
    return v


def minValueDepth(state, depth):
    if isTerminal(state):
        return getUtility(state) * 1000
    if depth == DEPTH_LIMIT:
        return evaluateHeuristic(state)

    v = float('inf')
    for action in getActions(state):
        v = min(v, maxValueDepth(getResult(state, action), depth + 1))
    return v
def minimaxDepthDecision(state):
    best_action = None
    best_value = float('-inf')
    for action in getActions(state):
        value = minValueDepth(getResult(state, action), 1)
        if value > best_value:
            best_value = value
            best_action = action
    return best_action, best_value
print("=== Task 4: Depth-Limited minimax ===")
initial = getInitialState()

for limit in [1, 2, 3]:
    DEPTH_LIMIT = limit
    action, value = minimaxDepthDecision(initial)
    print(f"  DEPTH_LIMIT = {limit}  |  Best Move: {action}  |  Heuristic Value: {value:>6}  |  Depth at evaluation: {limit}")

=== Task 4: Depth-Limited Minimax ===
  DEPTH_LIMIT = 1  |  Best Move: (1, 1)  |  Heuristic Value:      4  |  Depth at evaluation: 1
  DEPTH_LIMIT = 2  |  Best Move: (1, 1)  |  Heuristic Value:      1  |  Depth at evaluation: 2
  DEPTH_LIMIT = 3  |  Best Move: (1, 1)  |  Heuristic Value:     12  |  Depth at evaluation: 3


# Task 5: Alpha-Beta Pruning

In [6]:
nodes_visited = 0
nodes_pruned  = 0

def maxValueAB(state, alpha, beta):
    global nodes_visited, nodes_pruned
    nodes_visited += 1

    if isTerminal(state):
        return getUtility(state)
    v = float('-inf')
    for action in getActions(state):
        v = max(v, minValueAB(getResult(state, action), alpha, beta))
        if v >= beta:
            nodes_pruned += 1
            return v
        alpha = max(alpha, v)
    return v
def minValueAB(state, alpha, beta):
    global nodes_visited, nodes_pruned
    nodes_visited += 1
    if isTerminal(state):
        return getUtility(state)
    v = float('inf')
    for action in getActions(state):
        v = min(v, maxValueAB(getResult(state, action), alpha, beta))
        if v <= alpha:
            nodes_pruned += 1
            return v
        beta = min(beta, v)
    return v
def alphaBetaDecision(state):
    best_action = None
    best_value  = float('-inf')
    alpha = float('-inf')
    beta  = float('inf')

    for action in getActions(state):
        value = minValueAB(getResult(state, action), alpha, beta)
        if value > best_value:
            best_value  = value
            best_action = action
        alpha = max(alpha, best_value)
    return best_action, best_value
print("=== Task 5: Alpha-Beta Pruning ===")
initial = getInitialState()

# Standard Minimax (reuse Task 3)
nodes_explored = 0
mm_action, mm_value = minimaxDecision(initial)
mm_nodes = nodes_explored

# Alpha-Beta
nodes_visited = 0
nodes_pruned  = 0
ab_action, ab_value = alphaBetaDecision(initial)
ab_visited = nodes_visited
ab_pruned  = nodes_pruned

print(f"\n--- Alpha-Beta ---")
print(f"Best Move    : {ab_action}")
print(f"Value        : {ab_value}")
print(f"Nodes Visited: {ab_visited}")
print(f"Nodes Pruned : {ab_pruned}")

print()
print("+------------------+-----------------+---------------+--------------+--------------+")
print("| Algorithm        | Nodes Visited   | Nodes Pruned  | Optimal Move | Utility Value|")
print("+------------------+-----------------+---------------+--------------+--------------+")
print(f"| Minimax          | {mm_nodes:<15} | {'0':<13} | {str(mm_action):<12} | {mm_value:<12} |")
print(f"| Alpha-Beta       | {ab_visited:<15} | {ab_pruned:<13} | {str(ab_action):<12} | {ab_value:<12} |")
print("+------------------+-----------------+---------------+--------------+--------------+")

reduction = round((1 - ab_visited / mm_nodes) * 100, 1) if mm_nodes else 0
print(f"\nAlpha-Beta reduced node visits by {reduction}% vs standard Minimax.")

=== Task 5: Alpha-Beta Pruning ===

--- Alpha-Beta ---
Best Move    : (0, 0)
Value        : 0
Nodes Visited: 18296
Nodes Pruned : 8180

+------------------+-----------------+---------------+--------------+--------------+
| Algorithm        | Nodes Visited   | Nodes Pruned  | Optimal Move | Utility Value|
+------------------+-----------------+---------------+--------------+--------------+
| Minimax          | 549945          | 0             | (0, 0)       | 0            |
| Alpha-Beta       | 18296           | 8180          | (0, 0)       | 0            |
+------------------+-----------------+---------------+--------------+--------------+

Alpha-Beta reduced node visits by 96.7% vs standard Minimax.
